# 01 - Dataset Exploration

This notebook is responsible only for raw dataset exploration.

It checks:

- expected RAF-DB grayscale folder structure;
- train, validation and test split sizes;
- class balance using the project class order;
- basic image properties;
- representative raw image samples.

PyTorch loading and privacy-filter sanity checks are handled by `check_data_loader.py`.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.configs as configs_module
import src.eda_reporting as eda_reporting

configs_module = importlib.reload(configs_module)
eda_reporting = importlib.reload(eda_reporting)

RANDOM_SEED = 42

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", configs_module.DEFAULT_DATA_ROOT)
print("Expected classes:", list(configs_module.CLASS_NAMES))

## 1.1 Dataset Structure

The dataset should be available under `data/raw/balanced-raf-db-dataset-7575-grayscale` and organized into `train`, `val` and `test`.

In [ ]:
images_df, structure_df = eda_reporting.collect_dataset_inventory(
    configs_module.DEFAULT_DATA_ROOT
)

print(f"Total images found: {len(images_df):,}")
display(structure_df)

## 1.2 Class Balance

This is the main place where class distributions are inspected. The DataLoader sanity script only performs a short technical check.

In [ ]:
class_counts_pivot, balance_summary = eda_reporting.class_count_tables(images_df)

display(class_counts_pivot)
display(balance_summary)

fig = eda_reporting.plot_dataset_balance(images_df, class_counts_pivot)
plt.show()

## 1.3 Image File Properties

This checks file extensions, dimensions, channels and dtypes on a deterministic sample before training.

In [ ]:
file_extensions, shape_df = eda_reporting.inspect_image_properties(
    images_df,
    sample_size=1000,
    random_seed=RANDOM_SEED,
)

print("File extensions:")
display(file_extensions.to_frame())

print("Image shape summary:")
display(shape_df[["height", "width", "channels"]].describe().T)

print("Dtypes:")
display(shape_df["dtype"].value_counts(dropna=False).rename("count").to_frame())

print("Unreadable sampled images:", int((~shape_df["read_ok"]).sum()))

## 1.4 Raw Sample Preview

These are raw images only. Privacy-filter sanity checks are handled by `check_data_loader.py`.

In [ ]:
fig = eda_reporting.plot_raw_samples(
    images_df,
    split="train",
    samples_per_class=3,
    random_seed=RANDOM_SEED,
)
plt.show()

## 1.5 EDA Summary

- Dataset structure and class balance are checked here.
- DataLoader, privacy transformations and tensor batches are checked by `check_data_loader.py`.
- Training, evaluation and final plots are handled by the project scripts and later notebooks.